# EGFR Mutation Status Prediction from CT Radiomics
## CPTAC-LUAD Cohort (CTDC clinical data + IDC imaging)

**Goal:** Predict EGFR mutation status (mutant vs. wild-type) from CT radiomic features (first-order + GLCM) using a random forest classifier with 5-fold cross-validation.

**Cohort:** CPTAC-LUAD study, target N = 84 patients (as specified by the requester; not independently verified in this notebook draft — see verification cell below).

---

> **IMPORTANT CAVEATS — read before running**
> 1. The CTDC GraphQL field names below (`molecular_characterization`, `mutation_status`, etc.) are best-guess based on typical CTDC/CRDC data-model conventions. They have **not** been verified against the live schema in this session. Run the introspection cell first and correct field names as needed.
> 2. The IDC collection id `cptac_luad` is the expected lowercase-underscore form used by `idc-index`, but has **not** been confirmed against the live IDC index in this session.
> 3. Radiomic feature extraction requires segmentation masks (tumor ROI). Availability of pre-existing SEG/RTSTRUCT objects for this collection has **not** been verified. A fallback path is sketched but not implemented.
> 4. The figure of 84 patients is taken as given from the requester; this notebook queries the live cohort and reports the actual N obtained, which may differ.

## 1. Environment Setup

In [ ]:
# Install required packages (uncomment if running fresh)
# %pip install idc-index pyradiomics SimpleITK scikit-learn pandas numpy requests matplotlib seaborn

In [ ]:
import requests
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path

import SimpleITK as sitk
from radiomics import featureextractor

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, RocCurveDisplay

import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

## 2. CTDC Clinical Cohort Query

Queries the CTDC GraphQL API for CPTAC-LUAD cases with EGFR mutation status.

**NOTE:** Field names (`mutation_gene`, `mutation_status`, `molecular_characterization`, etc.) are unverified placeholders. Run the introspection cell first to confirm the actual schema before trusting query results.

In [ ]:
CTDC_API_URL = "https://datacommons.cancer.gov/ctdc/graphql"  # UNVERIFIED endpoint - confirm current URL

# --- Introspection: run this first to confirm actual schema field names ---
introspection_query = """
{
  __type(name: "case") {
    name
    fields {
      name
      type { name kind }
    }
  }
}
"""

resp = requests.post(CTDC_API_URL, json={"query": introspection_query})
print(resp.status_code)
try:
    print(json.dumps(resp.json(), indent=2))
except Exception as e:
    print("Could not parse response as JSON:", e)
    print(resp.text[:1000])

In [ ]:
# --- Main cohort query (placeholder field names - adjust after introspection above) ---
cohort_query = """
{
  case(study_short_name: "CPTAC-LUAD", first: 1000) {
    case_id
    submitter_id
    diagnoses {
      disease_type
      primary_diagnosis
    }
    molecular_characterization {
      gene_symbol
      mutation_status
    }
  }
}
"""

resp = requests.post(CTDC_API_URL, json={"query": cohort_query})
print(resp.status_code)

cohort_raw = resp.json()
# TODO: adjust parsing below once actual response structure is confirmed
cases = cohort_raw.get("data", {}).get("case", [])
print(f"Cases returned: {len(cases)}")

In [ ]:
# Flatten to a clinical dataframe with case_id and EGFR mutation label
records = []
for c in cases:
    egfr_status = None
    for mc in c.get("molecular_characterization", []) or []:
        if mc.get("gene_symbol", "").upper() == "EGFR":
            egfr_status = mc.get("mutation_status")
            break
    records.append({
        "case_id": c.get("case_id"),
        "submitter_id": c.get("submitter_id"),
        "egfr_status_raw": egfr_status,
    })

clinical_df = pd.DataFrame(records)

# TODO: confirm what raw values this field actually takes (e.g. "Mutant"/"Wildtype",
# "Yes"/"No", specific variant strings like "L858R"). Collapsing to binary here
# is a guess pending real data inspection.
clinical_df = clinical_df.dropna(subset=["egfr_status_raw"])
clinical_df["egfr_mutant"] = clinical_df["egfr_status_raw"].str.lower().str.contains(
    "mutant|positive|yes", na=False
).astype(int)

print(f"Cases with usable EGFR status: {len(clinical_df)}")
clinical_df.head()

## 3. IDC Imaging Pull (CT, `cptac_luad` collection)

Uses `idc-index` to enumerate CT series for the collection and joins on patient/case identifier to the CTDC cohort above.

**NOTE:** Collection id string and current IDC data release version are unverified.

In [ ]:
from idc_index import index

idc_client = index.IDCClient()

# Confirm collection id spelling/availability
all_collections = idc_client.get_collections()
matches = [c for c in all_collections if "luad" in c.lower() or "cptac" in c.lower()]
print("Candidate collections containing 'luad' or 'cptac':")
print(matches)

In [ ]:
COLLECTION_ID = "cptac_luad"  # confirm against `matches` output above; adjust if spelling differs

series_df = idc_client.sql_query(
    f"""
    SELECT PatientID, StudyInstanceUID, SeriesInstanceUID, Modality, collection_id
    FROM index
    WHERE collection_id = '{COLLECTION_ID}' AND Modality = 'CT'
    """
)

print(f"CT series found in {COLLECTION_ID}: {len(series_df)}")
print(f"Unique patients with CT in {COLLECTION_ID}: {series_df['PatientID'].nunique()}")
series_df.head()

In [ ]:
# Join CTDC clinical cohort to IDC imaging on patient identifier.
# TODO: confirm that CTDC's `submitter_id` (or case_id) actually matches IDC's PatientID
# format for CPTAC-LUAD. CPTAC identifiers are often consistent across DCs, but this
# has not been verified here - inspect a few values from both sides before trusting the join.

cohort_imaging = clinical_df.merge(
    series_df, left_on="submitter_id", right_on="PatientID", how="inner"
)

n_patients_final = cohort_imaging["PatientID"].nunique()
print(f"Patients with both CTDC EGFR status AND IDC CT imaging: {n_patients_final}")
print("Requester-specified target cohort size: 84 (unverified against live data)")
if n_patients_final != 84:
    print("WARNING: actual joined cohort size does not match the target of 84. "
          "Investigate identifier mismatches, missing mutation calls, or missing imaging "
          "before proceeding.")

## 4. Segmentation / ROI Check

Radiomic feature extraction requires a region of interest (tumor mask), not just the raw CT series. This step checks for existing SEG/RTSTRUCT objects in IDC for the matched series.

**NOTE:** Whether CPTAC-LUAD ships pre-existing tumor segmentations in IDC has not been verified in this session. If it does not, you will need a segmentation step (manual via 3D Slicer, or an auto-segmentation model) before radiomics can run.

In [ ]:
seg_df = idc_client.sql_query(
    f"""
    SELECT PatientID, StudyInstanceUID, SeriesInstanceUID, Modality, collection_id
    FROM index
    WHERE collection_id = '{COLLECTION_ID}' AND Modality IN ('SEG', 'RTSTRUCT')
    """
)

print(f"Segmentation objects found in {COLLECTION_ID}: {len(seg_df)}")
if len(seg_df) == 0:
    print(
        "No SEG/RTSTRUCT objects found for this collection. "
        "Radiomics extraction below CANNOT proceed without ROI masks. "
        "You'll need to either (a) locate masks from another source, "
        "(b) manually segment a sample, or (c) run an auto-segmentation "
        "tool (e.g., TotalSegmentator) as a preprocessing step - none of "
        "which is implemented in this notebook."
    )

## 5. Radiomic Feature Extraction (First-Order + GLCM)

Assumes CT volumes and matching ROI masks have been downloaded locally (see `download_idc_series` placeholder). Uses `pyradiomics`, restricted to `firstorder` and `glcm` feature classes only, per the requester's specification.

In [ ]:
# Configure pyradiomics extractor: first-order + GLCM only
extractor = featureextractor.RadiomicsFeatureExtractor()
extractor.disableAllFeatures()
extractor.enableFeatureClassByName("firstorder")
extractor.enableFeatureClassByName("glcm")

print("Enabled feature classes:", extractor.enabledFeatures)

In [ ]:
def download_idc_series(series_uid, dest_dir):
    """
    Placeholder for downloading a DICOM series from IDC and converting to a
    volume (e.g., via idc_client.download_from_selection + dicom2nifti or
    SimpleITK's ImageSeriesReader). Not implemented - wire up against your
    actual local storage/cache strategy.
    """
    raise NotImplementedError(
        "download_idc_series is a placeholder. Implement series download "
        "(e.g., idc_client.download_from_selection(seriesInstanceUID=...)) "
        "and DICOM-to-volume conversion before running feature extraction."
    )


def extract_features_for_patient(ct_path, mask_path, extractor):
    """Run pyradiomics on a single CT volume + mask pair."""
    image = sitk.ReadImage(ct_path)
    mask = sitk.ReadImage(mask_path)
    result = extractor.execute(image, mask)
    # Drop pyradiomics diagnostic keys, keep only actual feature values
    features = {k: v for k, v in result.items() if not k.startswith("diagnostics_")}
    return features

In [ ]:
# Feature extraction loop (skeleton - requires download_idc_series and
# segmentation masks to be in place per the caveats above).

feature_rows = []
errors = []

for _, row in cohort_imaging.drop_duplicates("PatientID").iterrows():
    patient_id = row["PatientID"]
    try:
        ct_path = DATA_DIR / f"{patient_id}_ct.nii.gz"      # placeholder path convention
        mask_path = DATA_DIR / f"{patient_id}_mask.nii.gz"  # placeholder path convention
        if not ct_path.exists() or not mask_path.exists():
            raise FileNotFoundError(
                f"Missing local CT/mask for {patient_id}. "
                "Run download + segmentation steps first."
            )
        feats = extract_features_for_patient(str(ct_path), str(mask_path), extractor)
        feats["PatientID"] = patient_id
        feature_rows.append(feats)
    except Exception as e:
        errors.append((patient_id, str(e)))

print(f"Features extracted for {len(feature_rows)} patients")
print(f"Errors/skipped: {len(errors)}")
if errors:
    print("First few errors:", errors[:3])

radiomics_df = pd.DataFrame(feature_rows)

## 6. Modeling: Random Forest with 5-Fold Cross-Validation

In [ ]:
# Merge features with labels
model_df = radiomics_df.merge(
    clinical_df[["submitter_id", "egfr_mutant"]],
    left_on="PatientID", right_on="submitter_id", how="inner"
)

print(f"Final modeling cohort: {len(model_df)} patients")
print(f"Class balance:\n{model_df['egfr_mutant'].value_counts()}")

feature_cols = [c for c in model_df.columns if c.startswith(("original_firstorder", "original_glcm"))]
X = model_df[feature_cols].values
y = model_df["egfr_mutant"].values

print(f"Feature matrix shape: {X.shape}")

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("rf", RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "roc_auc": "roc_auc",
    "accuracy": "accuracy",
    "f1": "f1",
}

cv_results = cross_validate(
    pipeline, X, y, cv=cv, scoring=scoring,
    return_train_score=False, n_jobs=-1
)

results_summary = pd.DataFrame({
    metric: cv_results[f"test_{metric}"] for metric in scoring
})
results_summary.index = [f"fold_{i+1}" for i in range(5)]

print(results_summary)
print("\nMean \u00b1 SD across folds:")
print(results_summary.agg(["mean", "std"]))

In [ ]:
# Feature importance (fit on full data for inspection purposes only - not used for CV metrics above)
pipeline.fit(X, y)
rf_model = pipeline.named_steps["rf"]

importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
importances.head(20).plot(kind="barh")
plt.gca().invert_yaxis()
plt.xlabel("Feature importance")
plt.title("Top 20 radiomic features (first-order + GLCM) - Random Forest")
plt.tight_layout()
plt.show()

## 7. Summary of Unverified Assumptions

Restated here for visibility before you act on any results from this notebook:

- **Cohort size (N=84):** taken as given from the requester; this notebook computes and reports the actual joined cohort size at runtime (Section 3) rather than assuming it matches.
- **CTDC GraphQL endpoint URL and schema field names:** placeholders based on typical CRDC conventions, not confirmed live. Run the introspection query (Section 2) first.
- **EGFR mutation status encoding:** binary mutant/wild-type collapse is a guess; actual CTDC values may include specific variants (L858R, exon19del, T790M, etc.) that may warrant separate handling.
- **IDC collection id `cptac_luad`:** unverified spelling/availability; confirmed via the `matches` query in Section 3 at runtime.
- **Patient ID join key (CTDC `submitter_id` vs. IDC `PatientID`):** assumed directly comparable; not verified. Inspect sample values from both sources before trusting the merge.
- **Segmentation/ROI availability:** not verified. If CPTAC-LUAD lacks shipped tumor masks in IDC, Section 5 cannot run as written and needs a segmentation step inserted first.
- **DICOM download and CT/mask conversion:** left as an unimplemented placeholder (`download_idc_series`) since the actual download/caching strategy depends on your storage environment (local disk, GCS, SBG CGC, etc.) which was not specified.